# Grayson Towncar Operations Intelligence Audit

Reproducible, read-only companion analysis for the executive report. The notebook
inspects operational tables, reconstructs eligible event cohorts, tests transparent
baselines, and writes the canonical aggregate `artifact.json` used by the HTML report.

In [1]:
from datetime import date
from pathlib import Path
import json
import sys

REPOSITORY_ROOT = Path.cwd().resolve()
DATABASE_PATH = "scratch/operations_intelligence/prod_snapshot.sqlite3"
ANALYSIS_CUTOFF = date(2026, 7, 31)
ALLOW_PRELIMINARY_FALLBACK = True

sys.path.insert(0, str(REPOSITORY_ROOT / "docs/operations-intelligence"))
import analysis

results = analysis.run_analysis(
    database_path=DATABASE_PATH,
    cutoff=ANALYSIS_CUTOFF,
    allow_preliminary=ALLOW_PRELIMINARY_FALLBACK,
)

## tl;dr

In [2]:
from IPython.display import Markdown, display

profile = results["source_profile"]
status = results["status_history"]
routes = results["route_metrics"]
route_baseline = results["route_baseline"]
rating = "Share with caveats" if profile["fresh_snapshot_present"] else "Needs revision"

display(Markdown(
    f'''
- **Validation status: {rating}.** `{profile['requested_path']}` was
  {'available and used' if profile['fresh_snapshot_present'] else 'not available; the preliminary working copy was used'}.
- **The eligible execution cohort begins {status['instrumentation_start']}.**
  Valid first-event status-chain coverage is **{analysis.format_percent(status['cohort_chain_rate'])}**
  across **{analysis.format_number(status['cohort_completed_legs'])}** completed legs.
- **Route history is useful for shadow testing, not production control.**
  **{routes['reliable_buckets']} of {routes['total_buckets']}** route buckets have at least five samples;
  the held-out historical-median baseline has MAE **{analysis.format_number(route_baseline['historical_median_mae_minutes'], 1)} min**
  versus **{analysis.format_number(route_baseline['static_mae_minutes'], 1)} min** for the static table.
- **Recommendation:** fix event integrity and decision snapshots first, then launch governed scorecards
  and shadow baselines before any predictive automation.
'''
))


- **Validation status: Needs revision.** `scratch/operations_intelligence/prod_snapshot.sqlite3` was
  not available; the preliminary working copy was used.
- **The eligible execution cohort begins 2026-02-01.**
  Valid first-event status-chain coverage is **60.2%**
  across **13,298** completed legs.
- **Route history is useful for shadow testing, not production control.**
  **115 of 456** route buckets have at least five samples;
  the held-out historical-median baseline has MAE **8.2 min**
  versus **9.2 min** for the static table.
- **Recommendation:** fix event integrity and decision snapshots first, then launch governed scorecards
  and shadow baselines before any predictive automation.


## Context & Methods

The decision is where to invest in operational data collection, analytics, and
automation. The audience is operational/product leadership. The source database is
opened with SQLite `mode=ro` and `PRAGMA query_only=ON`; no production credentials or
write statements are used.

### Key Assumptions

- Django timestamps are stored as UTC and converted to `America/New_York` before
  comparison with local service dates.
- Historical conclusions use an algorithmic instrumentation boundary: the first
  service month with at least 100 completed legs and at least 75% timely completion
  event capture.
- Status-chain validity follows the implemented bounds: on-way to picked-up is 1-180
  minutes, and picked-up to completed is 2-180 minutes.
- Route-duration baselines use in-house, non-excluded legs and a chronological 80/20
  train/test split. They are non-causal operational diagnostics.
- Published outputs contain aggregates only; no names, contacts, exact addresses,
  payment identifiers, or raw record extracts are emitted.

In [3]:
display(Markdown(
    "### Source fingerprint\n\n"
    + "\n".join([
        f"- Selected source: `{profile['selected_path']}` ({profile['source_tier']})",
        f"- SHA-256: `{profile['sha256']}`",
        f"- File modified (UTC): `{profile['modified_utc']}`",
        f"- SQLite quick-check: `{profile['quick_check']}`",
        f"- Tables: **{profile['table_count']}**; migrations: **{profile['migration_count']}**",
        f"- Latest recorded migration: `{profile['latest_migration']}`",
        f"- Cutoff/timezone: **{profile['analysis_cutoff']} / {profile['timezone']}**",
    ])
))

### Source fingerprint

- Selected source: `content/db.sqlite3` (preliminary working copy)
- SHA-256: `1b8fb83f23067e3a5bc044561d746aa39cb77a69ac7cb1838a4681c15dd5e5e5`
- File modified (UTC): `2026-07-31T20:54:14.687728+00:00`
- SQLite quick-check: `ok`
- Tables: **99**; migrations: **312**
- Latest recorded migration: `2026-07-30 02:47:33.990790`
- Cutoff/timezone: **2026-07-31 / America/New_York**

## Data

In [4]:
def markdown_table(rows, columns, limit=None):
    selected = rows[:limit] if limit else rows
    header = "| " + " | ".join(label for _, label in columns) + " |"
    divider = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for row in selected:
        values = []
        for field, _ in columns:
            value = row.get(field)
            if isinstance(value, float):
                value = f"{value:.3f}"
            values.append(str(value if value is not None else "—").replace("|", "/"))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, divider, *body])

display(Markdown(
    "### Core table inventory\n\n" + markdown_table(
        results["table_inventory"],
        [("domain", "Domain"), ("rows", "Rows"), ("first_record", "First"),
         ("latest_record", "Latest"), ("state", "State")],
    )
))

### Core table inventory

| Domain | Rows | First | Latest | State |
| --- | --- | --- | --- | --- |
| Reservations | 13934 | 2025-04-27 13:42:14 | 2026-07-27 18:05:48 | available |
| Legs | 24124 | 2026-01-10 15:29:09 | 2026-07-11 18:24:05 | available |
| Status events | 69212 | 2026-02-08 04:19:29 | 2026-07-11 20:37:59 | available |
| Flights | 25456 | 2025-12-04 05:31:32 | 2026-07-31 20:54:27 | available |
| Audit events | 260 | 2026-07-18 21:45:45 | 2026-07-31 20:15:08 | available |
| Schedule snapshots | 199 | 2026-02-10 23:55:19 | 2026-07-31 20:14:24 | available |
| Schedule snapshot entries | 10744 | — | — | available |
| Schedule drafts | 20 | 2026-06-11 22:05:20 | 2026-06-22 01:08:15 | available |
| Draft assignments | 923 | — | — | available |
| Draft events | 551 | — | — | available |
| Route timing buckets | 456 | 2026-02-10 22:31:25 | 2026-07-31 19:16:28 | available |
| Daily driver capacity | 0 | — | — | empty |
| Demand patterns | 0 | — | — | empty |
| Driver locations | 0 | — | — | empty |
| Drivers | 51 | — | — | available |
| Fleet vehicles | 14 | — | — | available |
| Driver-vehicle assignments | 1987 | — | — | available |
| Leg payments | 17730 | 2026-05-19 03:56:18 | 2026-07-07 18:57:44 | available |
| Customer payments | 14652 | 2025-04-14 15:11:07 | 2026-07-11 20:12:32 | available |
| Operational tasks | 7993 | 2026-03-15 13:46:56 | 2026-07-31 20:06:31 | available |
| Communication attempts | 511 | 2026-03-17 15:00:13 | 2026-07-31 18:20:38 | available |
| Staff activity | 201 | 2026-07-11 20:52:02 | 2026-07-31 21:07:13 | available |
| Email log | 10547 | 2026-04-03 23:08:31 | 2026-07-31 18:20:38 | available |
| Time-clock shifts | 70 | 2026-06-02 20:01:13 | 2026-07-11 20:43:14 | available |
| Follow-up tasks | 274 | 2026-07-27 19:27:55 | 2026-07-31 20:54:15 | available |
| Lead activities | 1541 | 2026-07-27 19:27:54 | 2026-07-31 20:54:15 | available |
| GHL sync logs | 1 | 2026-07-18 19:13:19 | 2026-07-18 19:13:19 | available |

In [5]:
display(Markdown(
    "### Data trust matrix\n\n" + markdown_table(
        results["trust_matrix"],
        [("data_asset", "Asset"), ("coverage", "Coverage"), ("trust", "Trust"),
         ("safe_use", "Safe use"), ("limitation", "Limitation")],
    )
))

### Data trust matrix

| Asset | Coverage | Trust | Safe use | Limitation |
| --- | --- | --- | --- | --- |
| Reservations and booked legs | 13,934 reservations; 24,124 legs | Conditional | booked demand and service mix after excluding invalid dates/statuses | mutable operational state and date outliers |
| Status history | usable cohort begins 2026-02-01; 69,212 events | Conditional | duration distributions after chain and timing validation | repeats, late backfills, and write-path bypasses |
| Flights | 25,456 flights; actual-time coverage 78.6% in cohort | Conditional | current flight operations and final actual-time availability | no historical forecast snapshots for prediction-at-decision-time |
| Schedule drafts and snapshots | 199 snapshots; 20 drafts | Conditional | recent schedule-change and publish-workflow analysis | short history and mutable pre-instrumentation decisions |
| Audit log | 2026-07-18 to 2026-07-31 | Unreliable historically | recent workflow spot checks only | recent start and incomplete write-path coverage |
| Route timing metrics | 456 buckets; 115 have at least 5 samples | Conditional | display estimates for sufficiently sampled, recently refreshed buckets | sparse/stale buckets and incomplete refresh triggers |
| Payments and payouts | 14,652 customer payments; 17,730 leg payments | Conditional | reconciliation when transaction records control summary fields | multiple representations of paid and refunded amounts |
| Staff and communications activity | activity since 2026-07-11; communications since 2026-03-17 | Conditional | recent workload and workflow adoption analysis | not a historical productivity series |
| Driver capacity and demand aggregates | 0 capacity rows; 0 demand rows | Missing | none until population and freshness checks exist | implemented models are not populated |
| Historical GPS trajectories | 0 driver-location rows | Missing | latest vehicle visibility only via FleetVehicle fields | no historical route, dwell, or deadhead reconstruction |

## Results

In [6]:
display(Markdown(
    "### Material data-quality findings\n\n" + markdown_table(
        results["quality_findings"],
        [("severity", "Severity"), ("finding", "Finding"), ("evidence", "Evidence"),
         ("analytical_risk", "Risk"), ("recommended_fix", "Fix")],
    )
))

### Material data-quality findings

| Severity | Finding | Evidence | Risk | Fix |
| --- | --- | --- | --- | --- |
| Critical | The required fresh production snapshot is absent | Analysis used content/db.sqlite3 as a preliminary working copy | headline values cannot be certified as production-current | place an immutable export at the requested path and rerun the notebook |
| High | Bulk completion bypasses status-event creation | reservation and leg admin actions use bulk update while driver/dispatcher paths create LegStatus rows | current status and event history diverge by workflow | route all transitions through one transactional state-change service |
| High | Capacity, demand, and GPS analytical tables are unpopulated | capacity=0, demand=0, driver locations=0 | utilization, supply/demand, deadhead, and route-path claims are not directly measurable | populate governed aggregates and retain consented telemetry only for defined uses |
| High | Completed-leg event history is incomplete outside the instrumented cohort | cohort begins 2026-02-01; timely completion-event rate is 95.4% | historic on-time and duration trends would be biased | publish metric eligibility windows and backfill only with explicit provenance |
| High | Flight forecasts are overwritten rather than snapshotted | Flight stores the latest scheduled, estimated, and actual times but no observation history | delay accuracy at assignment time and leakage-safe models cannot be reconstructed | append immutable forecast observations with observed_at and provider metadata |
| High | Route metrics are sparse, stale, and refreshed from selected paths only | 115 of 456 buckets have at least 5 samples; 185 are more than 14 days stale at cutoff | display estimates vary in quality and silently fall back to static assumptions | refresh from every completion path and enforce sample/freshness gates |
| High | Status events can repeat and earliest/latest semantics can disagree | 1,440 completed legs contain repeats; 189 change validity by semantic choice | route and service-duration metrics can change based on ORM ordering | define canonical occurred_at semantics and make transitions idempotent |
| Medium | Pickup dates contain implausible outliers | 2 legs fall outside 2015-2028; maximum is 3220-03-06 | unbounded queries and time series produce misleading ranges | validate service dates on write and quarantine existing outliers |
| Medium | Schedule and staff audit coverage is recent | audit starts 2026-07-18; staff activity starts 2026-07-11 | historical schedule churn and staff productivity cannot be compared consistently | define retention and event coverage SLAs before scorecard use |

In [7]:
monthly = [
    row for row in results["status_history"]["monthly"]
    if date.fromisoformat(row["month"]) >= results["status_history"]["instrumentation_start"]
]
display(Markdown(
    "### Status-chain coverage by service month\n\n" + markdown_table(
        monthly,
        [("month_label", "Month"), ("completed_legs", "Completed legs"),
         ("timely_completion_event_rate", "Timely completed event"),
         ("earliest_valid_chain_rate", "Valid first-event chain"),
         ("latest_valid_chain_rate", "Valid last-event chain"),
         ("four_status_coverage_rate", "Four-status coverage")],
    )
))

### Status-chain coverage by service month

| Month | Completed legs | Timely completed event | Valid first-event chain | Valid last-event chain | Four-status coverage |
| --- | --- | --- | --- | --- | --- |
| Feb 2026 | 1970 | 0.764 | 0.529 | 0.534 | 0.577 |
| Mar 2026 | 2583 | 0.964 | 0.605 | 0.612 | 0.703 |
| Apr 2026 | 2566 | 0.990 | 0.611 | 0.616 | 0.759 |
| May 2026 | 2688 | 0.995 | 0.634 | 0.637 | 0.835 |
| Jun 2026 | 2591 | 0.993 | 0.603 | 0.604 | 0.899 |
| Jul 2026 | 900 | 0.998 | 0.629 | 0.641 | 0.848 |

In [8]:
display(Markdown(
    "### Repeated status events\n\n" + markdown_table(
        results["status_history"]["repeat_summary"],
        [("status_label", "Status"), ("repeated_leg_status_pairs", "Repeated leg/status pairs"),
         ("extra_events", "Extra events"), ("max_occurrences", "Maximum occurrences")],
    )
    + f"\n\nExact same-timestamp duplicates: **{results['status_history']['exact_duplicate_events']}**. "
      f"First/last semantic validity disagreements: **{results['status_history']['semantic_disagreement']}**."
))

### Repeated status events

| Status | Repeated leg/status pairs | Extra events | Maximum occurrences |
| --- | --- | --- | --- |
| Confirmed | 2941 | 3672 | 12 |
| On The Way | 648 | 819 | 10 |
| Picked Up | 546 | 721 | 16 |
| On Location | 493 | 653 | 8 |
| In Progress | 508 | 642 | 5 |
| Completed | 477 | 613 | 7 |

Exact same-timestamp duplicates: **0**. First/last semantic validity disagreements: **189**.

In [9]:
display(Markdown(
    "### Route timing readiness\n\n" + markdown_table(
        results["route_metrics"]["confidence"],
        [("confidence_bucket", "Samples per bucket"), ("route_buckets", "Route buckets"),
         ("underlying_samples", "Underlying samples")],
    )
))

display(Markdown(
    "### Observed drive-time segments\n\n" + markdown_table(
        results["route_baseline"]["route_summary"],
        [("route", "Route"), ("trip_type", "Trip type"), ("samples", "Samples"),
         ("observed_median_minutes", "Median"), ("observed_p75_minutes", "P75"),
         ("static_minutes", "Static"), ("median_minus_static", "Median - static")],
        limit=20,
    )
))

### Route timing readiness

| Samples per bucket | Route buckets | Underlying samples |
| --- | --- | --- |
| 20+ samples | 41 | 4815 |
| 10-19 samples | 36 | 498 |
| 5-9 samples | 38 | 258 |
| 1-4 samples | 341 | 522 |

### Observed drive-time segments

| Route | Trip type | Samples | Median | P75 | Static | Median - static |
| --- | --- | --- | --- | --- | --- | --- |
| Disney Resort -> MCO Terminal | return | 2698 | 33.100 | 36.500 | 30 | 3.100 |
| MCO Terminal -> Disney Resort | arrival | 2325 | 35.700 | 44.000 | 30 | 5.700 |
| Port Canaveral Area -> MCO Terminal | cruise | 244 | 49.500 | 55.200 | 55 | -5.500 |
| MCO Terminal -> Other Hotel | arrival | 206 | 44.700 | 53.400 | 25 | 19.700 |
| Universal Resort -> MCO Terminal | return | 160 | 23.600 | 26.800 | 25 | -1.400 |
| Other Hotel -> MCO Terminal | return | 155 | 30.800 | 37.000 | 25 | 5.800 |
| MCO Terminal -> Other | arrival | 152 | 40.600 | 51.800 | 35 | 5.600 |
| MCO Terminal -> Universal Resort | arrival | 145 | 28.400 | 37.100 | 25 | 3.400 |
| Disney Resort -> SFB Terminal | return | 144 | 58.800 | 62.700 | 60 | -1.200 |
| Other -> MCO Terminal | return | 131 | 32.000 | 36.700 | 35 | -3.000 |
| SFB Terminal -> Disney Resort | arrival | 107 | 60.100 | 74.500 | 60 | 0.100 |
| MCO Terminal -> Port Canaveral Area | cruise | 89 | 54.400 | 66.400 | 55 | -0.600 |
| Disney Resort -> Port Canaveral Area | cruise | 59 | 74.600 | 81.300 | 72 | 2.600 |
| Port Canaveral Area -> Disney Resort | cruise | 56 | 69.800 | 80.600 | 72 | -2.200 |
| Disney Resort -> Universal Resort | other | 53 | 29.100 | 32.500 | 28 | 1.100 |
| Disney Resort -> Other | other | 50 | 35.400 | 46.800 | 35 | 0.400 |
| Universal Resort -> Disney Resort | other | 45 | 29.400 | 32.900 | 28 | 1.400 |
| Other -> Disney Resort | other | 29 | 33.800 | 59.000 | 35 | -1.200 |
| Other -> Port Canaveral Area | cruise | 27 | 65.400 | 74.000 | 35 | 30.400 |
| Other Hotel -> Port Canaveral Area | cruise | 25 | 58.300 | 64.300 | 55 | 3.300 |

In [10]:
ops = results["operations"]
demand = results["demand_baseline"]
flights = results["flights"]
display(Markdown(f'''
### Operational opportunity sizing

- **Assignment coverage:** {analysis.format_percent(ops['leg_summary']['assignment_rate'])}
  across {analysis.format_number(ops['leg_summary']['legs'])} eligible cohort legs.
- **Affiliate share of assigned work:** {analysis.format_percent(ops['leg_summary']['affiliate_share_of_assigned'])};
  this is a proxy, not a confirmed reactive farm-out rate.
- **Assignment lead-time proxy:** median {analysis.format_number(ops['assignment_lead_median_hours'], 1)} hours
  across {analysis.format_number(ops['assignment_lead_samples'])} legs; the field may represent the latest rather than first assignment.
- **Recent snapshot churn proxy:** {analysis.format_percent(ops['snapshot_change_rate'])}
  across {analysis.format_number(ops['snapshot_comparable_assignments'])} comparable assignment states.
- **Flight-linked arrival coverage:** {analysis.format_percent(flights['linked_flight_rate'])};
  actual-time coverage is {analysis.format_percent(flights['actual_time_rate'])} in the eligible cohort.
- **Demand baseline:** weekday-median MAE {analysis.format_number(demand['weekday_median_mae_legs'], 2)} legs/day
  versus {analysis.format_number(demand['overall_median_mae_legs'], 2)} for the overall median on
  {analysis.format_number(demand['test_days'])} held-out days.
'''))


### Operational opportunity sizing

- **Assignment coverage:** 93.0%
  across 15,132 eligible cohort legs.
- **Affiliate share of assigned work:** 21.2%;
  this is a proxy, not a confirmed reactive farm-out rate.
- **Assignment lead-time proxy:** median 15.6 hours
  across 14,054 legs; the field may represent the latest rather than first assignment.
- **Recent snapshot churn proxy:** 36.0%
  across 5,211 comparable assignment states.
- **Flight-linked arrival coverage:** 98.7%;
  actual-time coverage is 78.6% in the eligible cohort.
- **Demand baseline:** weekday-median MAE 22.50 legs/day
  versus 24.14 for the overall median on
  28 held-out days.


In [11]:
display(Markdown(
    "### KPI framework\n\n" + markdown_table(
        results["kpi_framework"],
        [("role", "Role"), ("metric", "Metric"), ("definition", "Definition"),
         ("cadence", "Cadence"), ("current_readiness", "Readiness"),
         ("required_work", "Required work")],
    )
))

### KPI framework

| Role | Metric | Definition | Cadence | Readiness | Required work |
| --- | --- | --- | --- | --- | --- |
| Primary KPI | On-time pickup rate | eligible legs with a verified actual pickup timestamp inside the agreed trip-type SLA divided by eligible legs | weekly with daily exception review | Not decision-ready | define pickup semantics/SLA; make status events idempotent and source-aware |
| Primary KPI | In-house coverage rate | eligible service legs assigned to active in-house drivers by the dispatch cutoff divided by eligible scheduled legs | daily and weekly | Conditional | define dispatch cutoff and distinguish intentional affiliates from reactive farm-outs |
| Primary KPI | Published-schedule reliability | published legs whose driver and vehicle remain unchanged through pickup divided by published eligible legs | weekly | Recent cohorts only | make publish snapshots and reassignment reason codes complete |
| Driver | Valid status-chain coverage | completed eligible legs with ordered on-way, picked-up, and completed events inside duration bounds | daily quality control | Conditional | standardize first/last/reversal semantics |
| Driver | Assignment lead time | hours from the canonical first committed driver assignment to scheduled pickup | weekly by trip type | Proxy only | retain first assignment separately from latest assignment |
| Driver | Clear-time prediction error | absolute difference between predicted and verified actual clear time on eligible legs | weekly by route/time bucket | Prototype-ready cohort | version predictions and retain the value used at scheduling time |
| Guardrail | Overlong duty-day rate | driver-days above the agreed raw/effective span threshold divided by worked driver-days | daily and weekly | Conditional | confirm actual duty boundaries and break-credit policy |
| Guardrail | Near-term uncovered-leg rate | eligible legs unassigned inside the agreed hours-to-pickup threshold divided by eligible near-term legs | continuous operations alert | Conditional | define intentional affiliate/unassigned exclusions |

In [12]:
display(Markdown(
    "### Predictive feature readiness\n\n" + markdown_table(
        results["predictive_matrix"],
        [("candidate", "Candidate"), ("label_quality", "Label quality"),
         ("prototype_result", "Prototype"), ("decision", "Decision"),
         ("next_gate", "Next gate")],
    )
))

### Predictive feature readiness

| Candidate | Label quality | Prototype | Decision | Next gate |
| --- | --- | --- | --- | --- |
| Flight delay risk at dispatch time | Missing historical forecast snapshots | Blocked | Do not model yet | retain provider observations and assignment-time features |
| Pickup delay risk | Pickup event meaning and SLA are not canonical | Blocked | Do not automate | define actual pickup and capture transition source/reversal |
| Drive-duration baseline | 7,150 eligible in-house status chains | historical-route MAE 8.2 min vs static 9.2 min on 1,430 held-out legs | Shadow-test only | fresh snapshot, route normalization, complete write-path refresh |
| Airport dwell baseline | 2,865 arrival legs with on-location to picked-up events | median 25.5 min; p75 40.6 min | Interpret as a workflow proxy | confirm what on-location means operationally |
| Daily demand forecast | 181 service days from eligible cohort | weekday-median MAE 22.50 legs vs overall-median 24.14 | Use as planning baseline after fresh rerun | add booking-as-of snapshots for lead-time-aware forecasts |
| Chained-trip feasibility | Status chains exist but lateness and decision-time inputs are incomplete | Conditional | Backtest rules, not production automation | version clear-time predictions and define lateness outcomes |
| Farm-out risk | Affiliate assignment is only a proxy for reactive farm-out | Conditional | Do not train on current label | capture assignment reason and decision timestamp |

In [13]:
display(Markdown(
    "### Prioritized roadmap\n\n" + markdown_table(
        results["roadmap"],
        [("phase", "Phase"), ("recommendation", "Recommendation"),
         ("impact", "Impact"), ("effort", "Effort"), ("priority_score", "Score"),
         ("confidence", "Confidence"), ("dependencies", "Dependencies"),
         ("risks_and_guardrails", "Risks and guardrails")],
    )
))

### Prioritized roadmap

| Phase | Recommendation | Impact | Effort | Score | Confidence | Dependencies | Risks and guardrails |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | Canonical transition service | 5 | 3 | 7.000 | High | Application transaction boundary and admin refactor. | Migration must preserve current staff workflows; add audit-only rollout first. |
| 1 | Flight forecast observation history | 5 | 3 | 7.000 | High | AeroAPI refresh path, retention policy, provider cost controls. | Bound refresh cadence and retain provider timestamps. |
| 1 | Immutable operational event contract | 5 | 3 | 7.000 | High | Canonical transition service and event taxonomy. | Avoid rewriting history; corrections must be additive. |
| 1 | Schedule decision snapshots and reason codes | 5 | 3 | 7.000 | High | Draft/snapshot workflow and assignment service. | Keep snapshots compact and distinguish preview from commitment. |
| 1 | Nightly data-quality gates | 4 | 2 | 6.000 | High | Stable metric eligibility rules and alert owner. | Use rate-based, late-arrival-aware thresholds. |
| 2 | Governed metric refresh pipeline | 4 | 3 | 5.000 | High | Canonical events and scheduled batch execution. | Do not serve low-sample buckets without fallback labels. |
| 2 | Weekly operations scorecard | 4 | 3 | 5.000 | Medium | At least four weeks of stable instrumentation. | No targets until definitions and baselines are validated. |
| 2 | Payment reconciliation checks | 3 | 2 | 4.000 | Medium | Canonical transaction precedence and exception ownership. | Never auto-correct money without review and audit. |
| 3 | Duration and demand shadow baselines | 4 | 3 | 5.000 | Medium | Fresh eligible cohorts and stored prediction-as-of values. | Shadow only until calibration and segment stability pass. |
| 3 | Dispatcher exception decision support | 4 | 4 | 4.000 | Medium | Validated baselines, UI design, alert-fatigue review. | Never hide static fallback or force an assignment. |
| 4 | Production predictive automation | 3 | 5 | 1.000 | Low | All prior phases, monitoring, rollback, and owner approval. | Do not automate pickup-delay, farm-out, or chain decisions yet. |

## Takeaways

1. Make operational state changes event-complete and idempotent before treating
   service timing as a trusted KPI.
2. Preserve schedule and flight information as immutable decision-time observations;
   latest-state fields cannot support leakage-safe retrospective prediction.
3. Populate derived route, capacity, and demand datasets through one governed refresh
   process with freshness and sample-size gates.
4. Establish a small weekly scorecard before setting targets. Use simple duration and
   demand baselines in shadow mode, retaining human override and explicit fallbacks.
5. Do not automate flight delay, pickup delay, chain feasibility, or farm-out decisions
   until the report's instrumentation and stakeholder-validation gates are satisfied.

In [14]:
artifact_path = REPOSITORY_ROOT / "docs/operations-intelligence/artifact.json"
notes_path = REPOSITORY_ROOT / "scratch/operations_intelligence/analysis_results.json"
analysis.write_outputs(results, artifact_path, notes_path)
display(Markdown(
    f"Canonical report input written to `{artifact_path.relative_to(REPOSITORY_ROOT)}`. "
    f"Aggregate working notes written to `{notes_path.relative_to(REPOSITORY_ROOT)}`."
))

Canonical report input written to `docs\operations-intelligence\artifact.json`. Aggregate working notes written to `scratch\operations_intelligence\analysis_results.json`.